In [19]:
%reload_ext autoreload
%autoreload 2

In [20]:
from housepriceprediction.config import DATA_RAW, TXT_FILE
import pandas as pd

df = pd.read_csv(DATA_RAW / "train.csv", keep_default_na=False, na_values=["NA"])
pd.set_option("display.max_columns", None)
# print(df["GarageType"].isna().sum())
# print(df.select_dtypes("float64").isna().sum())
# print(df.shape)
# print(len(df))
# print(df.head())
# print(df.dtypes.value_counts())
lines = TXT_FILE.read_text(encoding="utf-8").splitlines()
current_feature = None
features_with_na = list()
seen_elems = set()

for line in lines:
    if not line.strip():                 # empty line (or only spaces/tabs)
        continue
    is_header = line[0].isspace()
    if not is_header and ":" in line:
        current_feature = line.split(":")[0].strip()
    elif line.split()[0].upper() == "NA":
        if current_feature in seen_elems:
            continue
        seen_elems.add(current_feature)
        features_with_na.append(current_feature)

print("Number of unique features with NA:", len(features_with_na))
for index, feature in enumerate(features_with_na):
    print(f"Index {index}: {feature}")

print()

real_cols_with_na = set(df.columns[df.isna().any()])
cols_with_na = set(features_with_na)
replace_with_none = real_cols_with_na & cols_with_na
accidentals = real_cols_with_na - cols_with_na
no_na_in_training = cols_with_na - real_cols_with_na

print("--- 1. To replace with 'None' ---")
print(replace_with_none)
print("\n--- 2. Accidentals ---")
print(df[sorted(accidentals)].isna().sum())
print("\n--- 3. No NA in training ---")
print(no_na_in_training)

print()

# Check the correlation between YearBuilt and GarageYrBlt
# (values close to 1 confirm that garages are typically built alongside the house)
print(df["YearBuilt"].corr(df["GarageYrBlt"]))

# ==========================================
# Task 1: Check if MasVnrType and MasVnrArea have missing values in the same rows
# Build the missingness mask for each column and compare them element-wise.
# Then check if all comparisons are True. Expected result: True.
# ==========================================
result_1 = (df["MasVnrType"].isna() == df["MasVnrArea"].isna()).all()
print("\nTask 1 - MasVnrType & MasVnrArea mask match:", result_1)


# ==========================================
# Task 2: Same idea for GarageType and GarageYrBlt, but with different columns.
# Expected result: True.
# ==========================================
result_2 = (df["GarageType"].isna() == df["GarageYrBlt"].isna()).all()
print("\nTask 2 - GarageType & GarageYrBlt mask match:", result_2)


# ==========================================
# Task 3: How frequent is 'SBrkr' in the 'Electrical' column?
# value_counts() has a parameter that returns proportions instead of absolute counts.
# Expected result: approximately 0.914.
# ==========================================
sbrkr_freq = df["Electrical"].value_counts(normalize=True)["SBrkr"]
print("\nTask 3 - Electrical 'SBrkr' frequency:", sbrkr_freq)


# ==========================================
# Task 4: For how many houses with a garage is the garage year built identical to the house year built?
# This supports the 79% figure from the documentation.
# - Compare GarageYrBlt with YearBuilt using == (returns a True/False mask).
# - Note: For houses without a garage, GarageYrBlt is NaN, and NaN == anything evaluates to False.
#   Filter out houses without a garage first, then compare.
# - Trick: .mean() on a boolean mask directly gives the proportion of True values.
# Expected result: approximately 0.79.
# ==========================================
df_with_garage = df.dropna(subset=["GarageYrBlt"])
garage_eq_house_year = (df_with_garage["GarageYrBlt"] == df_with_garage["YearBuilt"]).mean()
print("\nTask 4 - Proportion of identical garage and house year built:", garage_eq_house_year)

Number of unique features with NA: 14
Index 0: Alley
Index 1: BsmtQual
Index 2: BsmtCond
Index 3: BsmtExposure
Index 4: BsmtFinType1
Index 5: BsmtFinType2
Index 6: FireplaceQu
Index 7: GarageType
Index 8: GarageFinish
Index 9: GarageQual
Index 10: GarageCond
Index 11: PoolQC
Index 12: Fence
Index 13: MiscFeature

--- 1. To replace with 'None' ---
{'BsmtExposure', 'GarageCond', 'BsmtFinType1', 'BsmtFinType2', 'GarageFinish', 'PoolQC', 'BsmtCond', 'MiscFeature', 'BsmtQual', 'FireplaceQu', 'GarageQual', 'Fence', 'Alley', 'GarageType'}

--- 2. Accidentals ---
Electrical       1
GarageYrBlt     81
LotFrontage    259
MasVnrArea       8
MasVnrType       8
dtype: int64

--- 3. No NA in training ---
set()

0.8256674841743417

Task 1 - MasVnrType & MasVnrArea mask match: True

Task 2 - GarageType & GarageYrBlt mask match: True

Task 3 - Electrical 'SBrkr' frequency: 0.914324880054832

Task 4 - Proportion of identical garage and house year built: 0.7897026831036983


In [ ]:
# Missing values: findings and decisions
#
# Reading the data
# By default, pandas treats the string `"None"` as a missing value.
# In this dataset `"None"` is a real category (e.g. `MasVnrType = None` means
# *no masonry veneer*), so the file is read with:
#
# `pd.read_csv(..., keep_default_na=False, na_values=["NA"])`
#
# This reduced the missing values in `MasVnrType` from 872 to 8.
#
# Columns where NA means "absent" (14)
# According to `data_description.txt`, `NA` in these columns means the house
# does not have that feature (no garage, no basement, no pool, etc.):
#
# `Alley`, `BsmtQual`, `BsmtCond`, `BsmtExposure`, `BsmtFinType1`, `BsmtFinType2`,
# `FireplaceQu`, `GarageType`, `GarageFinish`, `GarageQual`, `GarageCond`,
# `PoolQC`, `Fence`, `MiscFeature`
#
# **Decision:** replace NA with an explicit `"None"` category.
#
# Other columns with missing values
#
# | Column | Missing | Meaning | Decision |
# |---|---|---|---|
# | `GarageYrBlt` | 81 | houses without a garage | **drop**: 79% identical to `YearBuilt` (corr. 0.83) |
# | `LotFrontage` | 259 | genuinely unknown | impute with the **median per `Neighborhood`** |
# | `MasVnrType` | 8 | most likely no veneer | `"None"` |
# | `MasVnrArea` | 8 | same 8 houses | `0` |
# | `Electrical` | 1 | genuinely unknown | most frequent value (`SBrkr`) |
#
# Other preprocessing decisions
# - `Id` is an identifier, not a feature: exclude it from the model inputs.
# - `SalePrice` is the target: separate it from the features.
# - `MSSubClass` is a code, not a quantity: treat it as categorical.
# - `MoSold`: to be decided (numeric or categorical).
#
# > All imputation that learns from the data (medians, most frequent value)
# > will be done inside the preprocessing pipeline, fitted on training data only,
# > to avoid data leakage.